[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/16_cross_entropy.ipynb)

# 🟢 Easy: Cross-Entropy Loss

Implement **cross-entropy loss** from scratch.

$$\text{CE}(x, y) = -\log\frac{e^{x_y}}{\sum_j e^{x_j}}$$

### Signature
```python
def cross_entropy_loss(logits: Tensor, targets: Tensor) -> Tensor:
    # logits: (B, C) float, targets: (B,) long indices
    # Returns: scalar loss (mean over batch)
```

### Rules
- Do NOT use `F.cross_entropy` or `nn.CrossEntropyLoss`
- Must be numerically stable (use logsumexp trick)

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [2]:
import torch

In [5]:
def my_softmax(x: torch.Tensor, dim: int = -1) -> torch.Tensor:
    max_logit = torch.max(logits, dim=dim, keepdim=True).values
    _x = x - max_logit
    numerator = torch.exp(_x)
    denominator = torch.sum(numerator, dim=dim, keepdim=True)
    return  numerator / denominator

In [7]:
def my_log_softmax(logits: torch.Tensor, dim: int = -1) -> torch.Tensor:
    # Numerically stable log_softmax: x - logsumexp(x)

    # Find the maximum logit for each sample to subtract for numerical stability
    max_logit = torch.max(logits, dim=dim, keepdim=True).values

    # Subtract max_logit from all logits, then exponentiate
    # (x_j - max_k x_k)
    stabilized_logits = logits - max_logit
    # exp(x_j - max_k x_k)
    exp_stabilized_logits = torch.exp(stabilized_logits)

    # Sum the exponentiated stable logits
    # sum_j exp(x_j - max_k x_k)
    sum_exp_stabilized_logits = torch.sum(exp_stabilized_logits, dim=dim, keepdim=True)

    # Take the logarithm of the sum, and add back the max_logit
    # log(sum_j exp(x_j - max_k x_k)) + max_k x_k = log(sum_j exp(x_j))
    log_sum_exp = torch.log(sum_exp_stabilized_logits) + max_logit

    # log_softmax(x) = x - logsumexp(x)
    return logits - log_sum_exp

In [8]:
def cross_entropy_loss(logits: torch.Tensor, targets: torch.Tensor):
  # logits: (B, C) float, targets: (B,) long indices
  # Returns: scalar loss (mean over batch)

  # 1. Calculate log_probabilities using my_log_softmax for numerical stability.
  log_probs = my_log_softmax(logits, dim=1)

  # 2. Convert targets to one-hot encoding.
  num_classes = logits.shape[1]
  targets_one_hot = torch.nn.functional.one_hot(targets, num_classes=num_classes).float()

  # 3. Apply the one-hot cross-entropy formula: -sum(targets_one_hot * log_probs)
  # This is equivalent to selecting the log_prob corresponding to the true target for each sample.
  sample_losses = -torch.sum(targets_one_hot * log_probs, dim=1)

  # 4. Calculate the mean loss over the batch.
  return sample_losses.mean()

In [9]:
def cross_entropy_loss4(logits, targets):
  # logits: (B, C) float, targets: (B,) long indices
  # Returns: scalar loss (mean over batch)

  # 1. Extract x_y (the logit corresponding to the true class for each sample)
  # We use advanced indexing: for each row (sample) in the batch,
  # select the logit at the index specified by 'targets'.
  target_logits = logits[torch.arange(logits.shape[0]), targets]

  # 2. Manually calculate log(sum(exp(x))) across the class dimension (C) for numerical stability.
  # This is the 'Log-Sum-Exp' trick.

  # Find the maximum logit for each sample to subtract for numerical stability
  max_logit = torch.max(logits, dim=1, keepdim=True).values

  # Subtract max_logit from all logits, then exponentiate
  stabilized_logits = logits - max_logit
  exp_stabilized_logits = torch.exp(stabilized_logits)

  # Sum the exponentiated stable logits
  sum_exp_stabilized_logits = torch.sum(exp_stabilized_logits, dim=1)

  # Take the logarithm of the sum and add back the max_logit
  log_sum_exp_logits = torch.log(sum_exp_stabilized_logits) + max_logit.squeeze(1)

  # 3. Apply the cross-entropy formula: log(sum(exp(x))) - x_y
  # This gives the loss for each sample in the batch.
  loss_per_sample = log_sum_exp_logits - target_logits

  # 4. Calculate the mean loss over the batch as specified in the signature.
  mean_loss = loss_per_sample.mean()

  return mean_loss

In [10]:
# 🧪 Debug
logits = torch.randn(4, 10)
targets = torch.randint(0, 10, (4,))
print('Loss:', cross_entropy_loss(logits, targets))
print('Loss:', cross_entropy_loss4(logits, targets))
print('Ref: ', torch.nn.functional.cross_entropy(logits, targets))

Loss: tensor(3.2166)
Loss: tensor(3.2166)
Ref:  tensor(3.2166)


In [11]:
# ✅ SUBMIT
from torch_judge import check
check('cross_entropy')


🧪 Testing: Cross-Entropy Loss (Easy)
──────────────────────────────────────────────────
  ✅ [1/4] Matches F.cross_entropy (4.5ms)
  ✅ [2/4] Numerical stability (0.7ms)
  ✅ [3/4] Scalar output (0.3ms)
  ✅ [4/4] Gradient flow (1.9ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (7.3ms total)
  Progress saved. Run status() to see your dashboard.

